# Lab 4 — Two frameworks, one task, one decision

**~25 minutes · nothing to fill in**

You have now built the same Global Bank support desk twice: as a crew in Lab 1, and in ADK in Lab 3.
This lab runs both, side by side, on the same ticket. Then it asks you to choose one, and to say what
would change your mind.

That last step is the real work of this lab. The numbers help you check your argument.

## 1 · One problem, kept the same

Both desks get the same ticket, the same tool and the same three roles. **Only the framework
changes.** That is the only way the comparison means anything.

In [ ]:
import os, time
MODEL = os.environ["OPENAI_MODEL"]
BASE  = os.environ["OPENAI_BASE_URL"]
KEY   = os.environ["OPENAI_API_KEY"]

TICKETS = {
    "GB-T-4471": "My transfer of Rs 25,000 to my landlord failed twice, but my account was debited once.",
    "GB-T-4472": "I get 'invalid OTP' every time I log in on my new phone, since yesterday.",
    "GB-T-4473": "Please update my registered address. I have moved to Pune.",
    "GB-T-4474": "My savings account shows Rs 1,200 less than my passbook.",
}

# The four teams on the desk. They match the Global Bank services from Day 1.
TEAMS = ["Accounts", "Transactions", "Authentication", "Customer"]
TEAM_RULE = ("The category is the kind of problem, in a few words. "
             "The team must be one of: " + ", ".join(TEAMS) + ".")

TICKET_ID = "GB-T-4471"

RESULTS = {}

def record(name, seconds, requests, prompt, completion, answer):
    RESULTS[name] = {"s": round(seconds,1), "requests": requests, "prompt": prompt,
                     "completion": completion, "total": prompt+completion, "answer": answer}

def table():
    cols = ["s","requests","prompt","completion","total"]
    print(f"{'framework':<14}" + "".join(f"{c:>12}" for c in cols))
    for k,v in RESULTS.items():
        print(f"{k:<14}" + "".join(f"{v[c]:>12}" for c in cols))

print("ticket:", TICKET_ID, "-", TICKETS[TICKET_ID])

## 2 · The crew

In [ ]:
from crewai import LLM, Agent as CrewAgent, Task, Crew, Process
from crewai.tools import tool

@tool("ticket_lookup")
def crew_ticket_lookup(ticket_id: str) -> str:
    """Return the text of a Global Bank customer support ticket by its id."""
    return TICKETS.get(ticket_id, "not found")

crew_llm = LLM(model=MODEL, base_url=BASE, api_key=KEY, temperature=0)   # plain name

c_res = CrewAgent(role="Ticket Researcher", goal="State only the facts in the ticket",
                  backstory="You pull the raw ticket.", llm=crew_llm,
                  tools=[crew_ticket_lookup], allow_delegation=False)
c_cls = CrewAgent(role="Support Triage Analyst", goal="Classify and name the owning team",
                  backstory="You triage the Global Bank customer support desk.",
                  llm=crew_llm, allow_delegation=False)
c_wr  = CrewAgent(role="Response Drafter", goal="Write the first reply to the bank customer",
                  backstory="You never invent a timeline.", llm=crew_llm, allow_delegation=False)

crew = Crew(
    agents=[c_res, c_cls, c_wr],
    tasks=[Task(description=f"Retrieve ticket {TICKET_ID} and list its facts.",
                expected_output="A short bulleted list.", agent=c_res),
           Task(description=f"Classify it and name the owning team. {TEAM_RULE}",
                expected_output="'Category: <x>' and 'Team: <y>'.", agent=c_cls),
           Task(description="Draft a three-sentence first reply to the customer. Never invent a "
                            "timeline, and do not promise a refund or any action the facts do not support.",
                expected_output="Three sentences.", agent=c_wr)],
    process=Process.sequential, verbose=False,
)

s = time.time()
before = crew_llm.get_token_usage_summary()
crew_out = await crew.kickoff_async()          # in a notebook, always use the async version
# Do NOT use crew.usage_metrics: it adds the shared crew_llm's running total once per agent,
# so this three-agent crew would be counted three times. Take a snapshot of the model object instead.
u = crew_llm.get_token_usage_summary().delta_since(before)
record("CrewAI", time.time()-s, u.successful_requests, u.prompt_tokens, u.completion_tokens,
       str(crew_out).strip())
table()

## 3 · The ADK desk

The function below prints one line for each model call: which agent made it, and how many prompt
tokens that call read.

In [ ]:
from google.adk.agents import Agent, SequentialAgent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

def adk_model():
    return LiteLlm(model=f"openai/{MODEL}", api_base=BASE, api_key=KEY)   # prefix + api_base

def adk_ticket_lookup(ticket_id: str) -> dict:
    """Return the text of a Global Bank customer support ticket by its id."""
    return {"text": TICKETS.get(ticket_id, "not found")}

async def run_adk_desk(name, include_contents="default"):
    a_res = Agent(name="researcher", model=adk_model(), tools=[adk_ticket_lookup],
                  instruction="Retrieve the ticket and list only its facts.", output_key="facts")
    a_cls = Agent(name="classifier", model=adk_model(), output_key="triage",
                  include_contents=include_contents,
                  instruction="Using:\n{facts}\nGive 'Category: <x>' and 'Team: <y>'. " + TEAM_RULE)
    a_wr  = Agent(name="drafter", model=adk_model(), include_contents=include_contents,
                  instruction="Facts:\n{facts}\nTriage:\n{triage}\nDraft a three-sentence reply "
                              "to the bank customer. Never invent a timeline, and do not promise a "
                              "refund or any action the facts do not support.")

    svc = InMemorySessionService()
    runner = Runner(app_name="cmp", agent=SequentialAgent(name="desk", sub_agents=[a_res, a_cls, a_wr]),
                    session_service=svc)
    await svc.create_session(app_name="cmp", user_id="u", session_id=name)
    msg = types.Content(role="user", parts=[types.Part(text=f"Handle ticket {TICKET_ID}.")])

    s = time.time(); prompt = cands = calls = 0; final = None
    async for ev in runner.run_async(user_id="u", session_id=name, new_message=msg):
        usage = getattr(ev, "usage_metadata", None)
        if usage:                                   # one of these per model call
            calls  += 1
            prompt += usage.prompt_token_count or 0
            cands  += usage.candidates_token_count or 0
            print(f"  {ev.author:<11} prompt {usage.prompt_token_count:>5}")
        if ev.is_final_response() and ev.content and ev.content.parts:
            final = ev.content.parts[0].text
    record(name, time.time()-s, calls, prompt, cands, (final or "").strip())

await run_adk_desk("ADK")
table()

Look at the prompt size of each call above: **it grows**. Each sub-agent reads more than the one before
it. Keep that in mind for section 5. First, one change.

## 3b · Decide what each agent receives

By default, an ADK sub-agent receives the **whole session so far**:

- the customer's message,
- the researcher's tool call and its result,
- every earlier answer,
- *and* the state you put into its instruction.

So `{facts}` arrives twice.

`include_contents="none"` sends only the instruction and the state you put into it. It is the same
desk, with one argument changed:

In [ ]:
await run_adk_desk("ADK, none", include_contents="none")
table()

## 4 · Read all the answers

Numbers come first, but the output is half of the comparison. A framework that is cheaper but gives
worse answers is not really cheaper.

Read each reply as the bank customer would. Does it promise a refund, a date or an action that the
ticket does not support? Both drafters were told not to. A model can still do it, and a wrong promise
from a bank is a real problem.

In [ ]:
for name, r in RESULTS.items():
    print("=" * 72)
    print(f"{name}  -  {r['s']}s, {r['requests']} requests, {r['total']} tokens")
    print("=" * 72)
    print(r["answer"][:700])
    print()

## 5 · Where the difference comes from

Before you conclude anything, be clear about *why* the token counts differ. One framework is not simply
wasteful. The difference is in what each one puts in the prompt by default.

- **CrewAI** sends each agent its role, goal and backstory, the task, and **the outputs of the tasks
  before it**. It does not send the whole conversation. There is a lot of text, but it has a limit.
- **ADK by default** sends each sub-agent the **whole session history** plus the state you put into its
  instruction. Later agents read everything again, and `{facts}` travels twice. That is why its prompt
  sizes grew, and why the default run was the biggest row.
- **`include_contents="none"`** makes ADK send only what you chose. Compare its row with the default
  row. Same agents, one argument. Check that the answer is still as good.
- **The researcher makes two calls in both frameworks.** The tool call and the answer are separate
  round trips to the model. That is why neither desk makes only three requests.

**Counting has its own trap.** `crew.usage_metrics` adds a shared model's running total once per
agent. On this crew it reports several times what the gateway served. So the crew cell takes a
snapshot of `crew_llm` instead. ADK's count comes from the event stream: one `usage_metadata` per model
call.

So the fair comparison is about **defaults**. Neither framework is lean or wasteful by nature. Each
has a default for what it sends, and in ADK you can change it with one argument.

## 6 · Choose one

Answer these questions for a real piece of work you have in your team. Write the answers down. Writing
them is the point of this section.

1. **Do you know the order of the steps in advance?** If yes, neither framework's dynamic routing helps
   you. A graph or a plain function might be the better answer.
2. **Who needs to read this code in six months?** Roles and backstories read like job descriptions.
   Explicit runners and named state read like a program. Both are fine. Your team has a preference,
   and it matters.
3. **Where must you be able to inspect the hand-off?** ADK's `output_key` gives you a name to point at.
   A crew gives you context that you mostly trust.
4. **How much time can a reply take?** Look at the `s` column. Run the lab again before you believe it.
   On the same work, the seconds change more than the tokens do.
5. **What single fact would change your mind?** If you cannot name one, you have a preference, not a
   decision.

### What each framework costs you

| | CrewAI | ADK |
|---|---|---|
| **Setup for a small job** | Low: three kinds of object, then run | Higher: runner, session, event loop |
| **What each call carries** | Role, goal, backstory and earlier task outputs | Whole session history and your state, unless `include_contents="none"` |
| **Counting the tokens** | `crew.usage_metrics` over-counts a shared model. Take a snapshot of the `LLM` | One `usage_metadata` per model call in the event stream |
| **Seeing the hand-off** | Context, mostly hidden | `output_key` and `{name}`, visible |
| **Who decides the order** | You, or a manager LLM if you ask for one | You: `SequentialAgent` or `AgentTool` |
| **Running in a notebook** | `kickoff()` fails in Jupyter. Use `kickoff_async()` | Async first. No wrong version to call |

**Neither framework is the advanced one.** Pick the one whose costs you are willing to pay, and be
able to say why.

---

**You can now** put two frameworks on one problem, measure them, and defend your choice with numbers,
not with taste.